# 08 - A new data version: fewer rows, retrain, redeploy

Notebooks 01-06 built model v1 from the full 1,640-player extract. This notebook
walks the loop you repeat every time the data changes:

| Step | What happens | Tool |
| --- | --- | --- |
| 1 | Cut the raw extract to fewer rows | pandas |
| 2 | Record it as a new data version | DVC (`dvc add`) |
| 3 | Rebuild features + splits | DVC (`dvc repro`) |
| 4 | Push the data, commit the receipts | DVC + git |
| 5-6 | Train and evaluate on the new version | SageMaker + MLflow |
| 7 | Register a new model version, move `@staging` | MLflow registry |
| 8 | Compare with the previous model | MLflow |
| 9-10 | Deploy `@staging` (optionally production) | SageMaker endpoint |
| 11 | Trace the new model back to its data | `scripts/check-lineage.py` |
| 12 | Undo, if the new version is worse | git + DVC |

Nothing is lost along the way. The full extract stays in the DVC cache and the S3
remote under its old hash, and model v1 stays in the registry.

**Before you start**

- Notebooks 00-06 have run, so `.env` exists and `@staging` points at a model.
- `git status` is clean. `.venv/` and `.env` must be gitignored, otherwise every
  run is tagged `-dirty` and its lineage cannot be restored.
- Training (~5 min, `ml.m5.xlarge`) and deployment (~6-8 min) cost money, and the
  endpoint bills until `99-cleanup.ipynb` removes it.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value

from churn.config import Settings, load_params

params = load_params()
settings = Settings.load(params)
print("region:", settings.region, "| bucket:", settings.bucket)

In [ ]:
def run(cmd, check=True):
    """Run a shell command from the project root and echo its output."""
    print("$", cmd)
    proc = subprocess.run(cmd, shell=True, cwd=PROJECT_ROOT, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc

## 0. Check the starting point

Two things must hold before we touch the data:

- **The git tree is clean.** The commit we start from is the one we roll back to in
  step 12, and a dirty tree would leak into the new model's `git_commit` tag.
- **The working data matches `dvc.lock`.** Otherwise we would be cutting rows from
  a file that is not the version git says it is.

We also remember what `@staging` points at now, to compare against in step 8.

In [ ]:
from churn.dvc_meta import describe, git_commit
from churn.registry import resolve_alias

start_commit = git_commit()
print("starting commit:", start_commit)
print(describe())

if start_commit.endswith("-dirty"):
    run("git status --short", check=False)
    raise SystemExit(
        "Commit (or .gitignore) the changes above first. "
        "Training from a dirty tree gives the new model lineage tags you cannot restore."
    )

run("dvc status")

try:
    previous = resolve_alias("staging", settings=settings, params=params)
    print(f"\n@staging today -> v{previous.version} (run {previous.run_id})")
except Exception:
    previous = None
    print("\n@staging is not set yet; step 8 will have nothing to compare against")

## 1. Make the smaller dataset

We keep `N_ROWS` players and overwrite `data/raw/player_churn.csv` in place. That
is safe: DVC already holds the full file under its old hash, and step 12 brings it
back.

The sample is **stratified on the label**, so the churn rate stays the same. With
only ~8% churners, a plain random sample can leave the test split with a handful
of positives and a meaningless AUC. A fixed `SEED` makes the cut reproducible.

In [ ]:
import pandas as pd

N_ROWS = 1200   # players to keep
SEED = 42       # same seed -> same sample

raw_path = Path("data/raw/player_churn.csv")
target = params["project"]["target_col"]

full = pd.read_csv(raw_path)
if len(full) <= N_ROWS:
    raise SystemExit(
        f"{raw_path} already has {len(full)} rows. Re-running this notebook? "
        "Roll back first (step 12) or lower N_ROWS."
    )

small = (
    full.groupby(target, group_keys=False)
    .sample(frac=N_ROWS / len(full), random_state=SEED)
    .sort_index()
)
small.to_csv(raw_path, index=False)

print(f"rows      : {len(full)} -> {len(small)}")
print(f"churn rate: {full[target].mean():.3f} -> {small[target].mean():.3f}")

## 2. Record it as a new data version

`dvc add` hashes the new file, stores it in the cache and rewrites the small
pointer file git tracks. The diff below **is** the new data version: a new `md5`
and a new `size`. Notebook 01 turned on `core.autostage`, so the pointer is
already staged, which is why we diff against `HEAD`.

In [ ]:
run("dvc add data/raw/player_churn.csv")
run("git diff HEAD -- data/raw/player_churn.csv.dvc", check=False)

## 3. Rebuild the prepared data

The raw file feeds `features`, which feeds `split`, so `dvc repro` reruns both and
rewrites `dvc.lock` with new hashes for every output. Those hashes are what the new
model will be tagged with.

Check the split summary: the test fold is now smaller, and its positive rate should
still be close to the real churn rate (~0.085), not 0.5.

In [ ]:
run("dvc repro")

features_summary = json.loads(Path("reports/features_summary.json").read_text())
split_summary = json.loads(Path("reports/split_summary.json").read_text())
print(f"feature table : {features_summary['n_rows']} rows x {features_summary['n_cols']} cols")
for key in ("train_rows", "validation_rows", "test_rows", "test_positive_rate"):
    print(f"{key:14s}: {split_summary[key]}")

print("\n" + describe())

## 4. Push the data, then commit the receipts

Order matters: **data to S3 first, receipts to git second.** A commit whose
pointers reference data nobody can fetch cannot be reproduced.

After this cell the tree must be clean again. The new commit is what the training
run in step 5 is tagged with.

In [ ]:
run("dvc push")
run(
    "git add data/raw/player_churn.csv.dvc dvc.lock "
    "reports/split_summary.json reports/features_summary.json"
)
run(f'git commit -m "Data version: stratified {len(small)}-player sample (seed {SEED})"')
run(f"git tag data-{len(small)}-rows-seed{SEED}", check=False)  # a readable name; skipped if it exists

data_commit = git_commit()
print("\nnew data version committed at", data_commit)
if data_commit.endswith("-dirty"):
    run("git status --short", check=False)
    raise SystemExit("The tree is still dirty. Commit or ignore the files above before training.")

## 5. Train on the new version

The same call as notebook 03. `lineage_tags()` reads the `dvc.lock` and commit we
just made, and stamps them on the run before the job starts.

In [ ]:
from churn.dvc_meta import lineage_tags
from churn.training import train

print("this run will be tagged:")
for key, value in sorted(lineage_tags().items()):
    print(f"  {key:34s} {value}")

result = train(settings=settings, params=params, run_name=f"data-{len(small)}-rows")

print("\nMLflow run  :", result.run_id)
print("training job:", result.training_job_name)
print("metrics     :", json.dumps(result.metrics, indent=2))

## 6. Evaluate against the gate

Same gate as notebook 04 (`evaluate.auc_threshold` in `params.yaml`). With fewer
rows the test fold is small, so AUC is noisier. If the model misses the gate the
next cell stops here, which is the gate doing its job: raise `N_ROWS`, roll back
(step 12), or investigate before registering anything.

In [ ]:
from churn.evaluation import evaluate

evaluation = evaluate(
    run_id=result.run_id,
    model_data=result.model_data,
    settings=settings,
    params=params,
)
verdict = "PASS" if evaluation.passed else "FAIL"
print(f"test AUC {evaluation.auc:.4f}  vs gate {evaluation.threshold:.2f}  ->  {verdict}")

if not evaluation.passed:
    raise SystemExit("Model missed the AUC gate; not registering or deploying it.")

## 7. Register a new model version and move `@staging`

`register()` creates the next version (v2, v3, ...) and copies the lineage tags onto
it. **Do not change data or commit between step 5 and here**: registration reads
`dvc.lock` and the commit as they are now, not from the training run.

Then `@staging` moves to the new version. The previous version is not touched; an
alias is only a pointer.

In [ ]:
from churn.registry import ALIAS_CANDIDATE, ALIAS_STAGING, promote, register

model = register(
    run_id=result.run_id,
    settings=settings,
    params=params,
    alias=ALIAS_CANDIDATE,
    description=f"{len(small)}-player sample, test AUC {evaluation.auc:.4f}",
)
promote(version=model.version, alias=ALIAS_STAGING, settings=settings, params=params)
print(f"{model.name} v{model.version} registered; @candidate and @staging now point at it")

## 8. Compare with the previous model

Same code, different data. Each column is one model version; the `dvc.` rows show
the data each one was trained on, next to the score it got.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
client = MlflowClient()


def version_summary(version):
    tags = client.get_model_version(model.name, version).tags
    row = {"test_auc": tags.get("test_auc", "?"), "git_commit": tags.get("git_commit", "?")[:12]}
    row.update({key: value[:12] for key, value in sorted(tags.items()) if key.startswith("dvc.")})
    return row


columns = {}
if previous is not None:
    columns[f"v{previous.version} (before)"] = version_summary(previous.version)
columns[f"v{model.version} (new)"] = version_summary(model.version)
pd.DataFrame(columns)

## 9. Deploy `@staging`

~6-8 minutes. Updates the `player-churn-staging` endpoint in place to serve the
new version, then sends five rows from the new test split and compares the scores
with the true labels.

In [ ]:
from churn.deployment import deploy, smoke_test

staging = deploy(environment="staging", settings=settings, params=params)
print(f"endpoint {staging.endpoint_name} serving v{staging.model_version} (via {staging.source})")

rows = "\n".join(Path("data/processed/test_x.csv").read_text().splitlines()[:5])
scores = smoke_test(staging.endpoint_name, settings.region, rows)
labels = pd.read_csv("data/processed/test_y.csv", header=None).squeeze().head(5).tolist()
pd.DataFrame({"probability": [round(s, 4) for s in scores], "actual": labels})

## 10. Promote to production (optional)

Off by default. In CI this goes through the `churn-production` GitHub Environment
and its reviewers (notebook 07); doing it here bypasses that gate. Set
`PROMOTE_TO_PRODUCTION = True` only when you mean it.

In [ ]:
from churn.registry import ALIAS_PRODUCTION

PROMOTE_TO_PRODUCTION = False

if PROMOTE_TO_PRODUCTION:
    promote(version=model.version, alias=ALIAS_PRODUCTION, settings=settings, params=params)
    production = deploy(environment="production", settings=settings, params=params)
    print(f"endpoint {production.endpoint_name} serving v{production.model_version}")
    print("smoke test:", [round(s, 4) for s in smoke_test(production.endpoint_name, settings.region, rows)])
else:
    print(f"skipped; @production is unchanged. v{model.version} is on staging only.")

## 11. Trace the new model back to its data

The question every model must be able to answer. The commit printed here should be
the one from step 4, and the restore recipe should not carry a dirty-tree warning.

In [ ]:
run("python scripts/check-lineage.py --alias staging")

## 12. Undo: back to the full dataset (optional)

If the smaller version is worse, put the old receipts back and let DVC restore the
matching bytes, then point `@staging` at the previous model again. Nothing is
retrained. Rerun the deploy cell (step 9) afterwards to serve the old model.

Off by default. `start_commit` comes from step 0; after a kernel restart, find the
commit with `python scripts/check-lineage.py --version <old version>` instead.

In [ ]:
ROLL_BACK = False

if ROLL_BACK:
    run(
        f"git checkout {start_commit} -- data/raw/player_churn.csv.dvc dvc.lock "
        "reports/split_summary.json reports/features_summary.json"
    )
    run("dvc checkout")
    print(f"raw rows now: {len(pd.read_csv(raw_path))}")
    run(f'git commit -m "Back to the data version at {start_commit[:12]}"')
    if previous is not None:
        promote(version=previous.version, alias=ALIAS_STAGING, settings=settings, params=params)
        print(f"@staging -> v{previous.version}; rerun step 9 to serve it again")
else:
    print("skipped; the new data version stays checked out.")

---
## Next

The same loop from a terminal, as CI runs it:

```sh
dvc add data/raw/player_churn.csv && dvc repro && dvc push
git add data/raw/player_churn.csv.dvc dvc.lock reports/split_summary.json reports/features_summary.json
git commit -m "new data version"
python -m churn.training --run-name data-v2
python -m churn.evaluation --fail-below-threshold
python -m churn.registry --require-gate --alias staging
python -m churn.deployment --environment staging
```

A column change works the same way, minus `dvc add`: edit `features.keep_cols` in
`params.yaml`, then `dvc repro`, commit and retrain.

⚠️ The staging endpoint is billing by the hour. `99-cleanup.ipynb` removes it.